In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap

df = pd.read_csv('../SBAnational.csv', low_memory=False)
print(df.shape)
df.head()

(899164, 27)


,LoanNr_ChkDgt,Name,City,State,Zip,Bank,BankState,NAICS,ApprovalDate,ApprovalFY,...,RevLineCr,LowDoc,ChgOffDate,DisbursementDate,DisbursementGross,BalanceGross,MIS_Status,ChgOffPrinGr,GrAppv,SBA_Appv
0,1000014003,ABC HOBBYCRAFT,EVANSVILLE,IN,47711,FIFTH THIRD BANK,OH,451120,28-Feb-97,1997,...,N,Y,NaN,28-Feb-99,"$60,000.00",$0.00,P I F,$0.00,"$60,000.00","$48,000.00"
1,1000024006,LANDMARK BAR & GRILLE (THE),NEW PARIS,IN,46526,1ST SOURCE BANK,IN,722410,28-Feb-97,1997,...,N,Y,NaN,31-May-97,"$40,000.00",$0.00,P I F,$0.00,"$40,000.00","$32,000.00"
2,1000034009,"WHITLOCK DDS, TODD M.",BLOOMINGTON,IN,47401,GRANT COUNTY STATE BANK,IN,621210,28-Feb-97,1997,...,N,N,NaN,31-Dec-97,"$287,000.00",$0.00,P I F,$0.00,"$287,000.00","$215,250.00"
3,1000044001,"BIG BUCKS PAWN & JEWELRY, LLC",BROKEN ARROW,OK,74012,1ST NATL BK & TR CO OF BROKEN,OK,0,28-Feb-97,1997,...,N,Y,NaN,30-Jun-97,"$35,000.00",$0.00,P I F,$0.00,"$35,000.00","$28,000.00"
4,1000054004,"ANASTASIA CONFECTIONS, INC.",ORLANDO,FL,32801,FLORIDA BUS. DEVEL CORP,FL,0,28-Feb-97,1997,...,N,N,NaN,14-May-97,"$229,000.00",$0.00,P I F,$0.00,"$229,000.00","$229,000.00"


In [2]:
print(df.dtypes)
print("\n")
print(df.isnull().sum())

LoanNr_ChkDgt          int64
Name                  object
City                  object
State                 object
Zip                    int64
Bank                  object
BankState             object
NAICS                  int64
ApprovalDate          object
ApprovalFY            object
Term                   int64
NoEmp                  int64
NewExist             float64
CreateJob              int64
RetainedJob            int64
FranchiseCode          int64
UrbanRural             int64
RevLineCr             object
LowDoc                object
ChgOffDate            object
DisbursementDate      object
DisbursementGross     object
BalanceGross          object
MIS_Status            object
ChgOffPrinGr          object
GrAppv                object
SBA_Appv              object
dtype: object


LoanNr_ChkDgt             0
Name                     14
City                     30
State                    14
Zip                       0
Bank                   1559
BankState              1566
NAICS

In [3]:
print(df['MIS_Status'].value_counts())


MIS_Status
P I F     739609
CHGOFF    157558
Name: count, dtype: int64


In [4]:
# drop rows where MIS_Status is null
df = df[df['MIS_Status'].notna()]

# build binary default label
df['Default'] = (df['MIS_Status'] == 'CHGOFF').astype(int)

print(df['Default'].value_counts())
print(f"\nDefault rate: {df['Default'].mean():.1%}")

Default
0    739609
1    157558
Name: count, dtype: int64

Default rate: 17.6%


In [5]:
# clean dollar columns
dollar_cols = ['DisbursementGross', 'BalanceGross', 'ChgOffPrinGr', 'GrAppv', 'SBA_Appv']

for col in dollar_cols:
    df[col] = df[col].replace('[\$,]', '', regex=True).str.strip().astype(float)

print(df[dollar_cols].dtypes)
print("\n")
print(df[dollar_cols].head())

DisbursementGross    float64
BalanceGross         float64
ChgOffPrinGr         float64
GrAppv               float64
SBA_Appv             float64
dtype: object


   DisbursementGross  BalanceGross  ChgOffPrinGr    GrAppv  SBA_Appv
0            60000.0           0.0           0.0   60000.0   48000.0
1            40000.0           0.0           0.0   40000.0   32000.0
2           287000.0           0.0           0.0  287000.0  215250.0
3            35000.0           0.0           0.0   35000.0   28000.0
4           229000.0           0.0           0.0  229000.0  229000.0


In [6]:
# get columns
cols_to_keep = [
    'NAICS', 'State', 'Term', 'NoEmp', 'NewExist',
    'CreateJob', 'RetainedJob', 'FranchiseCode', 'UrbanRural',
    'RevLineCr', 'LowDoc', 'DisbursementGross', 'GrAppv',
    'SBA_Appv', 'ApprovalFY', 'Default'
]

df = df[cols_to_keep]
print(df.shape)
print("\n")
print(df.isnull().sum())

(897167, 16)


NAICS                   0
State                  13
Term                    0
NoEmp                   0
NewExist              134
CreateJob               0
RetainedJob             0
FranchiseCode           0
UrbanRural              0
RevLineCr            4520
LowDoc               2578
DisbursementGross       0
GrAppv                  0
SBA_Appv                0
ApprovalFY              0
Default                 0
dtype: int64


In [7]:
# drop rows with missing State or NewExist
df = df.dropna(subset=['State', 'NewExist'])

# fill remaining nulls
df['RevLineCr'] = df['RevLineCr'].fillna('N')
df['LowDoc'] = df['LowDoc'].fillna('N')

# check no nulls remain
print(df.isnull().sum())
print(f"\nRows remaining: {len(df):,}")

NAICS                0
State                0
Term                 0
NoEmp                0
NewExist             0
CreateJob            0
RetainedJob          0
FranchiseCode        0
UrbanRural           0
RevLineCr            0
LowDoc               0
DisbursementGross    0
GrAppv               0
SBA_Appv             0
ApprovalFY           0
Default              0
dtype: int64

Rows remaining: 897,020


In [8]:
# default rate by industry so top 15 NAICS codes
naics_default = (df.groupby('NAICS')['Default']
                 .agg(['mean', 'count'])
                 .rename(columns={'mean': 'default_rate', 'count': 'loan_count'})
                 .sort_values('loan_count', ascending=False)
                 .head(15))

naics_default['default_rate'] = naics_default['default_rate'].map('{:.1%}'.format)
print(naics_default)

       default_rate  loan_count
NAICS                          
0              8.3%      201662
722110        23.8%       27935
722211        25.2%       19431
811111        15.5%       14538
621210         4.0%       14034
624410        12.5%       10092
812112        25.4%        9178
561730        19.1%        8872
621310        13.9%        8715
812320        19.0%        7878
541110        10.3%        7054
445310        12.1%        6844
621111         8.0%        6708
721110         6.6%        6698
447110        17.5%        6617
